In [1]:
!ls

Untitled.ipynb


In [2]:
print("hello")


hello


In [3]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer.py


In [16]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    if random.random() < 0.05:
        return {
            
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(3000.01, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': 'elektronika',
            'hour': random.randint(0, 5),
            'timestamp': datetime.now().isoformat(),
        }
    else:
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(5.0, 3000.0), 2),
            'store': random.choice(sklepy),
            'category': random.choice(kategorie),
            'hour': random.randint(6, 23),
            'timestamp': datetime.now().isoformat(),
        }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']} | hour={tx['hour']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [15]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value

    if tx["amount"] > 3000:
        print(f'ALERT: {tx["tx_id"]} | {tx["amount"]:.2f} PLN | {tx["store"]} | {tx["category"]}')
# TWÓJ KOD
# Dla każdej wiadomości: sprawdź amount > 1000, jeśli tak — wypisz ALERT

Overwriting consumer_filter.py


In [12]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value

    if tx["amount"] > 3000:
        tx["risk_level"] = "HIGH"
    elif tx["amount"] > 1000:
        tx["risk_level"] = "MEDIUM"
    else:
        tx["risk_level"] = "LOW"

    print(tx)

Writing consumer_enrich.py


In [13]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

for message in consumer:
    tx = message.value
    store = tx["store"]
    amount = tx["amount"]

    # 1. Zwiększ liczbę transakcji dla sklepu
    store_counts[store] += 1

    # 2. Dodaj kwotę do sumy dla sklepu
    if store not in total_amount:
        total_amount[store] = 0
    total_amount[store] += amount

    msg_count += 1

    # 3. Co 10 wiadomości wypisz tabelę
    if msg_count % 10 == 0:
        print("\nSklep | Liczba | Suma | Średnia")
        print("-" * 40)

        for store in store_counts:
            count = store_counts[store]
            total = total_amount[store]
            avg = total / count
            print(f"{store} | {count} | {total:.2f} | {avg:.2f}")

Writing consumer_count.py


In [14]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {
    "count": 0,
    "revenue": 0.0,
    "min": float("inf"),
    "max": float("-inf")
})

msg_count = 0

for message in consumer:
    tx = message.value
    category = tx["category"]
    amount = tx["amount"]

    stats[category]["count"] += 1
    stats[category]["revenue"] += amount
    stats[category]["min"] = min(stats[category]["min"], amount)
    stats[category]["max"] = max(stats[category]["max"], amount)

    msg_count += 1

    if msg_count % 10 == 0:
        print("\nKategoria | Liczba | Przychód | Min | Max")
        print("-" * 55)

        for category, s in stats.items():
            print(
                f"{category} | {s['count']} | {s['revenue']:.2f} | "
                f"{s['min']:.2f} | {s['max']:.2f}"
            )

Writing consumer_stats.py


In [ ]:
#odpowiedzi
#1 Jeśli uruchomię consumer_filter.py po zakończeniu producenta, consumer może nie wyświetlić nic, bo będzie czekał na nowe wiadomości.
#2 Jeśli dwóch konsumentów ma tę samą group_id, Kafka podzieli wiadomości między nich. 
#  Każda wiadomość trafi tylko do jednego z nich w ramach tej samej grupy.
#3 Przetwarzanie bezstanowe analizuje każdą wiadomość osobno, bez pamiętania wcześniejszych danych. 
#  Przetwarzanie stanowe przechowuje informacje z poprzednich wiadomości, np. liczniki, sumy albo średnie.